# <p style='font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#E1F5FE; font-weight:bold; color:#0288D1; border:4px solid #0288D1; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🔎 Your Personal AI Resume Auditor 🚀</p>

<div style="
  background-color: #E3F2FD; 
  border-left: 6px solid #0277BD; 
  border-top: 6px solid #0277BD; 
  border-radius: 12px; 
  padding: 16px 22px; 
  font-size: 16px; 
  font-family: 'Signika Negative', sans-serif; 
  color: #01579B; 
  box-shadow: 0 4px 10px rgba(0, 0, 0, 0.15); 
  transition: transform 0.3s ease, box-shadow 0.3s ease;">
  
  <h4 style="
    font-size: 18px; 
    color: #0288D1; 
    font-weight: bold; 
    margin-bottom: 12px;">
    🚗 Stop Sending Weak Resumes : Let AI Tear It Apart Before Recruiters Do.
  </h4>

  <p style="
    margin: 0; 
    font-size: 15px; 
    line-height: 1.6; 
    color: #0D47A1;">
  </p>
</div>

In [1]:
import os
from dotenv import load_dotenv
from datetime import datetime

from typing import Literal, List
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader

# <p style='font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#E1F5FE; font-weight:bold; color:#0288D1; border:4px solid #0288D1; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🔎 Load API Key 🚀</p>

In [2]:
# LOAD API KEY 

load_dotenv()
API_KEY = os.getenv(key = 'GEMINI_API_KEY')
llm = ChatGoogleGenerativeAI(model="models/gemma-3-27b-it", temperature=0.3)

print(f'Connected to GEMINI API : {API_KEY[:5]}***')

Connected to GEMINI API : AIzaS***


# <p style='font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#E1F5FE; font-weight:bold; color:#0288D1; border:4px solid #0288D1; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🔎 Build Schema 🚀</p>

In [3]:
# CREATE SCHEMA

class Suggestion(BaseModel):
    severity: Literal["HIGH", "MEDIUM", "LOW"]
    recommendation: str = Field(description="Clear actionable improvement suggestion")

class ResumeCritique(BaseModel):
    suggestions: List[Suggestion]

# DEFINE LLM PARSER (FORMAT INSTRUCTION FOR LLM)
parser = PydanticOutputParser(pydantic_object=ResumeCritique)

parser

PydanticOutputParser(pydantic_object=<class '__main__.ResumeCritique'>)

# <p style='font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#E1F5FE; font-weight:bold; color:#0288D1; border:4px solid #0288D1; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🔎 Define Gemini Model 🚀</p>

In [4]:
# DEFINE GEMINI MODEL

llm = ChatGoogleGenerativeAI(model = 'models/gemma-3-27b-it', temperature = 0.1)

llm

ChatGoogleGenerativeAI(profile={}, google_api_key=SecretStr('**********'), model='models/gemma-3-27b-it', temperature=0.1, client=<google.genai.client.Client object at 0x0000022E913FF2C0>, default_metadata=(), model_kwargs={})

# <p style='font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#E1F5FE; font-weight:bold; color:#0288D1; border:4px solid #0288D1; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🔎 Create Prompt 🚀</p>

In [5]:
# CREATE A PROMPT

# DEFINE CURRENT DATE
current_date = datetime.now().strftime(format = "%B %Y")

# BUILD INTERNAL PROMPT
PROMPT = PromptTemplate(template = """You are a brutally strict senior technical recruiter performing a full resume writing audit.
                                    Current date: {current_date}
                                    Your task: Critique EVERY writing weakness in the resume.
                            
                                    Rules: - Do NOT limit the number of suggestions. 
                                           - Scan section by section.
                                           - Scan bullet by bullet.
                                           - If something is weak, vague, unquantified, structurally wrong, or unclear → critique it.
                                           - If something is already strong and correct → do NOT invent criticism.
                                           - Continue listing suggestions until no further writing improvements can be made.
                        
                                    Focus strictly on: - PRAQ structure
                                                       - Quantification gaps
                                                       - Weak verbs
                                                       - Missing business impact
                                                       - Structural ordering issues
                                                       - Section misplacement
                                                       - ATS clarity
                                                       - Formatting inconsistencies
                                                       - Redundant wording
                                                       - Vague technical descriptions
                                                       - Overly long bullets
                                                       - Missing role clarity
                                                       - Skill misprioritization
                                    DO NOT: - Give career advice
                                            - Talk about recruiters reviewing code
                                            - Suggest generic improvements
                                    Severity definition: HIGH = blocks hiring decision
                                                         MEDIUM = important improvement
                                                         LOW = polish
                                    Format strictly: 
                                    HIGH 
                                    <recommendation>
                        
                                    HIGH
                                    <recommendation>
                        
                                    MEDIUM
                                    <recommendation>
                                    
                                    LOW
                                    <recommendation>
                        
                                    Continue until fully audited.
                                    Resume: {resume_text} 
                                    {format_instructions}""", 

                        # EXTERNAL VARIABLE
                        input_variables = ["resume_text"], 
                        partial_variables = {"format_instructions" : parser.get_format_instructions(),
                                             "current_date" : current_date})

# <p style='font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#E1F5FE; font-weight:bold; color:#0288D1; border:4px solid #0288D1; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🔎 Data Ingestion : Upload PDF File 🚀</p>

In [6]:
# PDF LOADER FUNCTION

def load_resume(pdf_path : str):

    # LOAD PDF DOCUMENT
    pdf_loader = PyPDFLoader(pdf_path)

    # LOAD ALL PAGES IN PDF DOCUMENT
    pages = pdf_loader.load()

    return "\n".join([page.page_content for page in pages])

# <p style='font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#E1F5FE; font-weight:bold; color:#0288D1; border:4px solid #0288D1; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🔎 Build Pipeline 🚀</p>

In [7]:
# CREATE FUNCTION TO CRITIQUE A RESUME WITH LLM

def critique_resume(pdf_path : str):

    # LOAD RESUME/CV
    resume_text = load_resume(pdf_path)

    # INJECT RESUME INTO PROMPT
    injected_prompt = PROMPT.format(resume_text=resume_text)

    # CRITIQUE WITH GEMINI LLM
    response = llm.invoke(injected_prompt)

    parsed = parser.parse(response.content)
    return parsed

In [8]:
# OUTPUT FORMAT (EXPECTED OUTPUT)

def format_output(critique: ResumeCritique):

    print("\nSuggestion for Improvement\n")

    # SORT SEVERITY BY DESCENDING
    severity_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
    sorted_suggestions = sorted(critique.suggestions, key=lambda x: severity_order[x.severity])

    # DISPLAY OUTPUT
    for s in sorted_suggestions:
        print(f"{s.severity}")
        print(f"{s.recommendation}\n")

# <p style='font-family: https://fonts.google.com/share?selection.family=Signika+Negative:wght@300..700; background-color:#E1F5FE; font-weight:bold; color:#0288D1; border:4px solid #0288D1; border-radius:12px; box-shadow: 0px 4px 12px rgba(0, 0, 0, 0.2); padding:10px; text-align:center; transition: all 0.3s ease;'>🔎 Test Model 🚀</p>

In [9]:
# TEST MODEL

result = critique_resume(r"C:\Python Data Analysis\LLM\AI Resume Critique\cv_computer-vision.pdf")
format_output(result)


Suggestion for Improvement

HIGH
Contact information lacks standardization. Include a professional email address (avoiding Gmail) and a consistently formatted phone number. Remove extraneous links (Kaggle, GitHub, personal site) from the header; these belong in the Projects section as supporting evidence.

HIGH
The 'AI Engineer Program Scholarship Awardee' entry is insufficient. This is an experience, not just a certification. Expand this to detail the program's duration, key skills learned, and any projects completed *within* the program. Use the PRAQ structure.

HIGH
Projects section lacks clear dates. '2026' for multiple projects is ambiguous. Specify start and end dates (e.g., Jan 2026 - Feb 2026) for each project to demonstrate a timeline of work.

HIGH
Quantify impact across *all* projects. 'Improving 25% mAP Score' is good, but needs context. 25% improvement *compared to what*? What was the previous mAP score? What was the business value of this improvement (e.g., reduced false